In [0]:
import mlflow.sklearn
import pandas as pd
import numpy as np
import pyspark.sql.functions as F
from delta.tables import DeltaTable

# get URI model. 
run_id = "157824302138461ea930bde970143be1"
model_uri = f"runs:/{run_id}/weather_profile_model"

print(f"loading model from: {model_uri}")
loaded_model = mlflow.sklearn.load_model(model_uri)
training_cols = loaded_model.feature_names_in_


In [0]:
# dinamic date range based on next 90 days from current date
dates = spark.sql("SELECT explode(sequence(current_date(), date_add(current_date(), 89), interval 1 day)) as date")

# list of municipalities and counties
municipality_codes = spark.table("dbw_routemind_euskadi_dev.bronze.municipality_codes").select("countyId", "municipalityCode").distinct()

# join dates with municipality codes
future_dates_mun = dates.crossJoin(municipality_codes)

In [0]:
future_dates_mun = future_dates_mun.toPandas()

# transformations(Feature Engineering Idéntico al Entrenamiento)
future_dates_mun['date'] = pd.to_datetime(future_dates_mun['date'])
future_dates_mun['day_of_year'] = future_dates_mun['date'].dt.dayofyear
future_dates_mun['day_sin'] = np.sin(2 * np.pi * future_dates_mun['day_of_year'] / 365.0)
future_dates_mun['day_cos'] = np.cos(2 * np.pi * future_dates_mun['day_of_year'] / 365.0)

# unique geographical key and geographic transformation
future_dates_mun['unique_municipality'] = future_dates_mun['countyId'].astype(str) + "_" + future_dates_mun['municipalityCode'].astype(str)
future_dates_mun = pd.get_dummies(future_dates_mun, columns=['unique_municipality'], drop_first=True)

In [0]:
# align inference columns with training columns in order to fill with 0 in case that a municipality is not in the list
for col in training_cols:
    if col not in future_dates_mun.columns:
        future_dates_mun[col] = 0

In [0]:
X_future = future_dates_mun[training_cols]

In [0]:
# Probabilistic Forecasting
probability = loaded_model.predict_proba(X_future)
future_dates_mun['prob_outdoor'] = probability[:, 1] # keep Outdoor probability

In [0]:
# columns to keep in target table
output_cols = ['date', 'countyId', 'municipalityCode', 'prob_outdoor']
predictions = spark.createDataFrame(future_dates_mun[output_cols])

In [0]:

#### Hierarchical Allocation (county average for municipalities with no historical data)

county_avg = predictions.filter(F.col("prob_outdoor") > 0).groupBy("date", "countyId").agg(
    F.round(F.avg("prob_outdoor"), 4).alias("prob_outdoor_county")
)

# final dataframe with sorted columns
final_gold = predictions.join(
    county_avg, on=["date", "countyId"], how="left"
).withColumn(
    "scoring_outdoor",
    F.coalesce(F.col("prob_outdoor"), F.col("prob_outdoor_county"))
).select("date", "countyId", "municipalityCode", "scoring_outdoor")


In [0]:
target_table = "dbw_routemind_euskadi_dev.gold.weather_prediction_scoring"
delta_path = "abfss://gold@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/weather_prediction/data"

final_gold.write \
    .format("delta") \
    .option("path", delta_path) \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(target_table)


print(f"overwrite completed on {target_table}. rows processed: {final_gold.count()}")